<a href="https://colab.research.google.com/github/Peeyusj/FashionMNIST/blob/main/FashionMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [3]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size,shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [6]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
train(train_dataloader, model, loss_fn, optimizer)

loss: 2.321667  [   64/60000]
loss: 2.296836  [ 6464/60000]
loss: 2.286338  [12864/60000]
loss: 2.257879  [19264/60000]
loss: 2.256848  [25664/60000]
loss: 2.230575  [32064/60000]
loss: 2.210040  [38464/60000]
loss: 2.192158  [44864/60000]
loss: 2.170204  [51264/60000]
loss: 2.174542  [57664/60000]


In [8]:
def test(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.eval()
    with torch.no_grad():
      for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [10]:
for epoch in range(5):
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn, optimizer)

loss: 1.910833  [   64/60000]
loss: 1.826347  [ 6464/60000]
loss: 1.853074  [12864/60000]
loss: 1.796578  [19264/60000]
loss: 1.715740  [25664/60000]
loss: 1.669279  [32064/60000]
loss: 1.707569  [38464/60000]
loss: 1.602271  [44864/60000]
loss: 1.635461  [51264/60000]
loss: 1.561958  [57664/60000]
loss: 1.533669  [   64/10000]
loss: 1.557366  [ 6464/10000]
loss: 1.509989  [   64/60000]
loss: 1.516759  [ 6464/60000]
loss: 1.413986  [12864/60000]
loss: 1.484691  [19264/60000]
loss: 1.460327  [25664/60000]
loss: 1.447492  [32064/60000]
loss: 1.333855  [38464/60000]
loss: 1.410639  [44864/60000]
loss: 1.183459  [51264/60000]
loss: 1.188156  [57664/60000]
loss: 1.268819  [   64/10000]
loss: 1.310348  [ 6464/10000]
loss: 1.284730  [   64/60000]
loss: 1.226141  [ 6464/60000]
loss: 1.255239  [12864/60000]
loss: 1.206175  [19264/60000]
loss: 1.314080  [25664/60000]
loss: 1.221662  [32064/60000]
loss: 1.128775  [38464/60000]
loss: 1.117384  [44864/60000]
loss: 1.216159  [51264/60000]
loss: 1.07